# Forecast Evaluation: Prophet

Evaluates the Prophet temperature forecasting model using
historical train/test backtesting.

For each city:
- Uses historical weather data for training
- Holds out the most recent observations as test data
- Generates predictions for the test period
- Compares predicted vs actual temperatures
- Calculates MAE and RMSE

In [0]:
%pip install prophet
dbutils.library.restartPython()

In [0]:
%run ./00_setup_config

In [0]:
# ============================================================
# 2. IMPORTS
# ============================================================
import mlflow
import mlflow.prophet
from prophet import Prophet
import pandas as pd
import numpy as np
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [0]:
# ============================================================
# 3. LOAD CURATED WEATHER DATA
# ============================================================
df_curated = spark.table("internship_databricks_ws.default.weather_curated")
print(f"Loaded {df_curated.count()} rows from weather_curated")

In [0]:
# ============================================================
# 4. GET CITIES + CITY_ID LOOKUP
#    (df_dims_city is the source of truth for city_id, not weather_curated)
# ============================================================
df_dims_city = spark.table("internship_databricks_ws.default.weather_dims_city")

city_id_lookup = {
    row["city"]: row["city_id"]
    for row in df_dims_city.select("city", "city_id").distinct().collect()
}
cities = list(city_id_lookup.keys())

print(f"Evaluating {len(cities)} cities:")
print(cities)

In [0]:
# ============================================================
# 5. CREATE / LOAD MLFLOW EXPERIMENT
# ============================================================
mlflow.set_experiment("/Shared/weather_forecasting_evaluation")

In [0]:
# ============================================================
# 6. EVALUATE EACH CITY (unchanged logic — just add city_id to results)
# ============================================================
evaluation_results = []

for city in cities:
    print("\n" + "=" * 60)
    print(f"Evaluating {city}...")
    print("=" * 60)

    city_pd = (
        df_curated
        .filter(col("city") == city)
        .select("weather_time_pkt", "temperature_c")
        .orderBy("weather_time_pkt")
        .toPandas()
    )

    city_pd["ds"] = pd.to_datetime(city_pd["weather_time_pkt"], errors="coerce")
    city_pd["y"] = pd.to_numeric(city_pd["temperature_c"], errors="coerce")
    city_pd = city_pd.dropna(subset=["ds", "y"])
    city_pd = city_pd.sort_values("ds").drop_duplicates(subset=["ds"], keep="last").reset_index(drop=True)

    if len(city_pd) < 48:
        print(f"Skipping {city} — only {len(city_pd)} usable rows (minimum required: 48)")
        continue

    split_index = int(len(city_pd) * 0.8)
    train = city_pd.iloc[:split_index][["ds", "y"]].copy()
    test = city_pd.iloc[split_index:][["ds", "y"]].copy()

    print(f"{city}: {len(train)} training rows, {len(test)} test rows")
    print(f"Training period: {train['ds'].min()} → {train['ds'].max()}")
    print(f"Testing period:  {test['ds'].min()} → {test['ds'].max()}")

    with mlflow.start_run(run_name=f"evaluation_{city}"):
        model = Prophet(daily_seasonality=False, weekly_seasonality=False, yearly_seasonality=False)
        model.fit(train)

        future = test[["ds"]].copy()
        forecast = model.predict(future)
        predictions = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()

        comparison = test.merge(predictions, on="ds", how="inner")

        if len(comparison) == 0:
            print(f"No matching test predictions found for {city}")
            continue

        mae = mean_absolute_error(comparison["y"], comparison["yhat"])
        rmse = np.sqrt(mean_squared_error(comparison["y"], comparison["yhat"]))

        mlflow.log_param("city", city)
        mlflow.log_param("total_rows", len(city_pd))
        mlflow.log_param("training_rows", len(train))
        mlflow.log_param("test_rows", len(test))
        mlflow.log_metric("MAE", float(mae))
        mlflow.log_metric("RMSE", float(rmse))

        evaluation_results.append({
            "city": city,
            "city_id": city_id_lookup[city],   # <-- NEW: tag city_id directly here
            "total_rows": len(city_pd),
            "training_rows": len(train),
            "test_rows": len(test),
            "MAE": float(mae),
            "RMSE": float(rmse)
        })

    print(f"{city}: MAE={mae:.2f}°C, RMSE={rmse:.2f}°C")

In [0]:
# ============================================================
# 7. CREATE FINAL RESULTS DATAFRAME
# ============================================================
df_evaluation = spark.createDataFrame(evaluation_results)

print("=" * 60)
print("FORECAST EVALUATION RESULTS")
print("=" * 60)
df_evaluation.orderBy("MAE").show(20, truncate=False)

In [0]:
# ============================================================
# 8. OVERALL PERFORMANCE
# ============================================================
overall_metrics = df_evaluation.agg(
    F.avg("MAE").alias("Average_MAE"),
    F.avg("RMSE").alias("Average_RMSE"),
    F.min("MAE").alias("Best_MAE"),
    F.max("MAE").alias("Worst_MAE")
)

print("=" * 60)
print("OVERALL PERFORMANCE")
print("=" * 60)
overall_metrics.show(truncate=False)

In [0]:


# Create Gold table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_forecast_evaluation
USING DELTA
LOCATION '{gold_path}weather_forecast_evaluation/'
""")

# Write evaluation results
df_evaluation.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "internship_databricks_ws.default.weather_forecast_evaluation"
    )

print("Forecast evaluation results saved successfully.")